# 🔍 Phân tích CORPUS của 15 Pires Requêtes

**Mục đích**: Xem nội dung thực tế của corpus để hiểu tại sao NDCG lại = 0

## 📦 Setup

In [1]:
import pandas as pd
import json
from pathlib import Path
import re

# Configuration
DATA_DIR = Path("../data")
SUBMISSION_FILE = "./6_last_version/submission_final.csv"
QRELS_FILE = DATA_DIR / "MultiHeirtt_qrels.tsv"
QUERIES_FILE = DATA_DIR / "multiheirtt_queries.jsonl" / "queries.jsonl"
CORPUS_FILE = DATA_DIR / "multiheirtt_corpus.jsonl" / "corpus.jsonl"

print("✅ Configuration")

✅ Configuration


## 📂 Charger données

In [2]:
# Charger corpus
corpus = {}
with open(CORPUS_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        doc = json.loads(line)
        corpus[doc['_id']] = {
            'title': doc.get('title', ''),
            'text': doc.get('text', '')
        }

# Charger qrels
qrels_df = pd.read_csv(QRELS_FILE, sep='\t')
qrels = {}
for _, row in qrels_df.iterrows():
    if row['query_id'] not in qrels:
        qrels[row['query_id']] = {}
    qrels[row['query_id']][row['corpus_id']] = row['score']

# Charger queries
queries = {}
with open(QUERIES_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        q = json.loads(line)
        queries[q['_id']] = q['text']

# Charger results
sub_df = pd.read_csv(SUBMISSION_FILE)
results = {}
for _, row in sub_df.iterrows():
    if row['query_id'] not in results:
        results[row['query_id']] = []
    results[row['query_id']].append(row['corpus_id'])

print(f"✅ Corpus: {len(corpus)} documents")

✅ Corpus: 10475 documents


## 📊 Tính NDCG

In [3]:
import numpy as np

def compute_ndcg(relevant_docs, retrieved_docs, k=10):
    retrieved_k = retrieved_docs[:k]
    dcg = sum(relevant_docs.get(doc_id, 0) / np.log2(i + 2) for i, doc_id in enumerate(retrieved_k))
    ideal_scores = sorted(relevant_docs.values(), reverse=True)[:k]
    idcg = sum(score / np.log2(i + 2) for i, score in enumerate(ideal_scores))
    return dcg / idcg if idcg > 0 else 0.0

# Tính NDCG cho mỗi query
query_scores = []
for query_id in qrels.keys():
    if query_id in results:
        ndcg = compute_ndcg(qrels[query_id], results[query_id])
        query_scores.append({
            'query_id': query_id,
            'query_text': queries.get(query_id, ''),
            'ndcg_10': ndcg,
            'relevant_docs': list(qrels[query_id].keys()),
            'retrieved_docs': results[query_id][:10]
        })

df = pd.DataFrame(query_scores)
print(f"✅ {len(df)} queries analyzed")

✅ 292 queries analyzed


---
## 🔴 PHÂN TÍCH CHI TIẾT: 15 PIRES REQUÊTES

Xem nội dung thực tế của corpus tương ứng

In [4]:
# Lấy 15 pires
worst_15 = df.nsmallest(15, 'ndcg_10')

print("="*100)
print("🔴 PHÂN TÍCH 15 PIRES REQUÊTES - XEM NỘI DUNG CORPUS")
print("="*100)

# Phân tích từng query
for idx, (row_idx, row) in enumerate(worst_15.iterrows(), 1):
    query_id = row['query_id']
    query_text = row['query_text']
    relevant_docs = row['relevant_docs']
    
    print(f"\n\n{'='*100}")
    print(f"#{idx} QUERY ID: {query_id}")
    print(f"    NDCG@10: {row['ndcg_10']:.4f}")
    print(f"    TEXT: {query_text}")
    print(f"\n    📚 CÓ {len(relevant_docs)} RELEVANT DOCUMENTS")
    
    # Hiển thị nội dung các relevant docs
    for doc_id in relevant_docs[:2]:  # Show first 2 relevant docs
        doc = corpus.get(doc_id, {})
        title = doc.get('title', 'N/A')
        text = doc.get('text', 'N/A')
        
        print(f"\n    📄 Document ID: {doc_id}")
        print(f"       Title: {title}")
        print(f"       Text length: {len(text)} chars")
        print(f"       Text preview (first 500 chars):")
        print(f"       {text[:500]}")
        print(f"       ...")

🔴 PHÂN TÍCH 15 PIRES REQUÊTES - XEM NỘI DUNG CORPUS


#1 QUERY ID: q8115b172
    NDCG@10: 0.0000
    TEXT: what is the pre-tax aggregate net unrealized loss in 2008?

    📚 CÓ 3 RELEVANT DOCUMENTS

    📄 Document ID: d8115b2c6
       Title: 
       Text length: 3764 chars
       Text preview (first 500 chars):
       AMERICAN TOWER CORPORATION AND SUBSIDIARIES NOTES TO CONSOLIDATED FINANCIAL STATEMENTS—(Continued) of certain of its assets and liabilities under its interest rate swap agreements held as of December 31, 2006 and entered into during the first half of 2007.
In addition, the Company paid $8.0 million related to a treasury rate lock agreement entered into and settled during the year ended December 31, 2008.
The cost of the treasury rate lock is being recognized as additional interest expense over t
       ...

    📄 Document ID: d8115b370
       Title: 
       Text length: 4312 chars
       Text preview (first 500 chars):
       TELEFLEX INCORPORATED NOTES?TO CONSOLIDATED FIN

---
## 🔬 PHÂN TÍCH ĐẶC ĐIỂM CỦA CORPUS

In [5]:
def analyze_corpus_text(text):
    """Phân tích đặc điểm của text"""
    return {
        'độ_dài_text': len(text),
        'số_từ': len(text.split()),
        'số_dòng': text.count('\n') + 1,
        'chứa_bảng': '|' in text or '\t' in text,
        'số_lượng_con_số': len(re.findall(r'\b\d+(?:\.\d+)?\b', text)),
        'số_lượng_năm': len(re.findall(r'\b(19|20)\d{2}\b', text)),
        'chứa_tiền_tệ': bool(re.search(r'\$|USD|EUR|£', text)),
        'chứa_phần_trăm': len(re.findall(r'\d+\.?\d*\s*%', text)),
        'chứa_dấu_phẩy_tách': ',' in text,
    }

# Phân tích corpus của pires queries
worst_corpus_features = []

for _, row in worst_15.iterrows():
    for doc_id in row['relevant_docs']:
        doc = corpus.get(doc_id, {})
        text = doc.get('text', '')
        features = analyze_corpus_text(text)
        features['query_id'] = row['query_id']
        features['doc_id'] = doc_id
        worst_corpus_features.append(features)

worst_corpus_df = pd.DataFrame(worst_corpus_features)

print("="*100)
print("🔬 ĐẶC ĐIỂM TRUNG BÌNH CỦA CORPUS TRONG 15 PIRES QUERIES")
print("="*100)
print(worst_corpus_df.drop(['query_id', 'doc_id'], axis=1).mean())

print(f"\n\n📊 CHI TIẾT:")
for col in worst_corpus_df.drop(['query_id', 'doc_id'], axis=1).columns:
    print(f"\n{col}:")
    print(f"  Min: {worst_corpus_df[col].min():.0f}")
    print(f"  Max: {worst_corpus_df[col].max():.0f}")
    print(f"  Mean: {worst_corpus_df[col].mean():.1f}")

🔬 ĐẶC ĐIỂM TRUNG BÌNH CỦA CORPUS TRONG 15 PIRES QUERIES
độ_dài_text           3746.339286
số_từ                  612.428571
số_dòng                 28.071429
chứa_bảng                0.589286
số_lượng_con_số         78.517857
số_lượng_năm            14.410714
chứa_tiền_tệ             0.892857
chứa_phần_trăm           5.607143
chứa_dấu_phẩy_tách       0.982143
dtype: float64


📊 CHI TIẾT:

độ_dài_text:
  Min: 175
  Max: 9847
  Mean: 3746.3

số_từ:
  Min: 23
  Max: 1708
  Mean: 612.4

số_dòng:
  Min: 1
  Max: 94
  Mean: 28.1

chứa_bảng:
  Min: 0
  Max: 1
  Mean: 0.6

số_lượng_con_số:
  Min: 0
  Max: 415
  Mean: 78.5

số_lượng_năm:
  Min: 0
  Max: 69
  Mean: 14.4

chứa_tiền_tệ:
  Min: 0
  Max: 1
  Mean: 0.9

chứa_phần_trăm:
  Min: 0
  Max: 41
  Mean: 5.6

chứa_dấu_phẩy_tách:
  Min: 0
  Max: 1
  Mean: 1.0


---
## 🔍 SO SÁNH VỚI 15 MEILLEURES QUERIES

In [ ]:
# Lấy 15 meilleures
best_15 = df.nlargest(15, 'ndcg_10')

# Phân tích corpus của best queries
best_corpus_features = []

for _, row in best_15.iterrows():
    for doc_id in row['relevant_docs']:
        doc = corpus.get(doc_id, {})
        text = doc.get('text', '')
        features = analyze_corpus_text(text)
        features['query_id'] = row['query_id']
        features['doc_id'] = doc_id
        best_corpus_features.append(features)

best_corpus_df = pd.DataFrame(best_corpus_features)

print("="*100)
print("🔍 SO SÁNH: PIRES vs MEILLEURES")
print("="*100)

comparison = pd.DataFrame({
    '❌ PIRES': worst_corpus_df.drop(['query_id', 'doc_id'], axis=1).mean(),
    '✅ MEILLEURES': best_corpus_df.drop(['query_id', 'doc_id'], axis=1).mean(),
})
comparison['DIFFÉRENCE'] = comparison['✅ MEILLEURES'] - comparison['❌ PIRES']

print(comparison.round(2).to_string())

print(f"\n\n💡 KEY INSIGHTS:")
for col in comparison.index:
    worst_val = comparison.loc[col, '❌ PIRES']
    best_val = comparison.loc[col, '✅ MEILLEURES']
    diff = comparison.loc[col, 'DIFFÉRENCE']
    
    if diff != 0:
        direction = "CÓ NHIỀU HƠN" if diff > 0 else "CÓ ÍT HƠN"
        print(f"\n{col}: Pires {direction} Meilleures ({diff:.1f})")
        print(f"    Pires: {worst_val:.1f} | Meilleures: {best_val:.1f}")

---
## 📋 NHẬN XÉT VỀ CẤU TRÚC CORPUS

In [ ]:
print("="*100)
print("📋 NHẬN XÉT VỀ CẤU TRÚC CORPUS")
print("="*100)

# Xem chi tiết một vài corpus
print("\n\n🔴 VÍ DỤ 1: CORPUS TỪ PIRES QUERY")
print("-" * 100)

worst_query = worst_15.iloc[0]
worst_doc_id = worst_query['relevant_docs'][0]
worst_doc = corpus.get(worst_doc_id, {})

print(f"Query: {worst_query['query_text']}")
print(f"Doc ID: {worst_doc_id}")
print(f"Title: {worst_doc.get('title', 'N/A')}")
print(f"\nText:")
print(worst_doc.get('text', '')[:1500])
print("...")

print("\n\n✅ VÍ DỤ 2: CORPUS TỪ MEILLEURES QUERY")
print("-" * 100)

best_query = best_15.iloc[0]
best_doc_id = best_query['relevant_docs'][0]
best_doc = corpus.get(best_doc_id, {})

print(f"Query: {best_query['query_text']}")
print(f"Doc ID: {best_doc_id}")
print(f"Title: {best_doc.get('title', 'N/A')}")
print(f"\nText:")
print(best_doc.get('text', '')[:1500])
print("...")

print("\n\n🎯 KẾT LUẬN:")
print("-" * 100)
print("""
Thế nào là corpus có tính chất làm NDCG thấp?

1. PHỨC TẠP: Corpus chứa quá nhiều dòng, bảng, con số
2. KHÔNG CÓ STRUCTURE: Văn bản dài mà không có tiêu đề rõ ràng
3. NHIỀU PHẦN TỬ: Chứa nhiều năm, tiền tệ, phần trăm nhưng khó tìm
4. NGÔN NGỮ PHỨC TẠP: Tên cột bảng dài, diễn đạt phức tạp
""")